<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-18-Pandas-Challenge-Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO

df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Rows: {len(df)}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units: {total_units:,}")

# Interpretation: 400 orders brought in $8,520.00 across 783 units, about $10.88 per unit sold.

Rows: 400
Total Revenue: $8,520.00
Total Units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (
    df.groupby('category', as_index = False)
    .agg(revenue=('revenue', 'sum'), orders = ('revenue', 'size'))
    .sort_values('revenue', ascending = False)
    .reset_index(drop = True)
)

by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)
display(by_category)

best, worst = by_category.iloc[0], by_category.iloc[-1]

# Interpretation: Food is the top category at $4,293.00 (50.4% of revenue), while RainGear is the smallest at $901.50 (10.6%).


,category,revenue,orders,share_pct
0,Food,4293.0,186,50.4
1,Merch,1771.5,79,20.8
2,Drink,1554.0,89,18.2
3,RainGear,901.5,46,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_order_revenue = 'mean', order_count = 'count', std = 'std')
    .round(2)
    .sort_values('avg_order_revenue', ascending = False)
)

by_vendor['std_error'] = (by_vendor['std'] / np.sqrt(by_vendor['order_count'])).round(2)
display(by_vendor)

top = by_vendor['avg_order_revenue'].idxmax()

# Answer: V-01 has the highest average order revenue at $22.60 over 94 orders.


,avg_order_revenue,order_count,std,std_error
vendor_id,,,,
V-01,22.60,94,17.61,1.82
V-18,21.75,108,17.82,1.71
V-05,20.58,93,15.23,1.58
V-10,20.31,105,17.78,1.74


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = merch_revenue / df['revenue'].sum() * 100

print(f"Merch Share of Revenue: {merch_share:.1f}%")

# Interpretation: Merch made $1,771.50 out of $8,520.00 total, so it is about 20.8% of sales. This makes it the second-biggest
# category after Food.


Merch Share of Revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

joined_table = df.merge(vendor_names, on = 'vendor_id', how = 'left', validate = 'many_to_one', indicator = True)

unmatched_ids = joined_table.loc[joined_table['_merge'] == 'left_only', 'vendor_id'].unique()
unmatched_orders = (joined_table['_merge'] == 'left_only').sum()
unmatched_revenue = joined_table.loc[joined_table['_merge'] == 'left_only', 'revenue'].sum()

print(f"Unmatched Vendor Id: {list(unmatched_ids)}")
print(f"Unmatched Order: {unmatched_orders}")
print(f"Unmatched Revenue: ${unmatched_revenue:,.2f} ({unmatched_revenue / df['revenue'].sum() * 100:.1f}% of total)")

joined_table['vendor_name'] = joined_table['vendor_name'].fillna('Unknown (' + joined_table['vendor_id'] + ')')
joined_table = joined_table.drop(columns = '_merge')

# Proof my left merge didn't change the row count or revenue total
assert len(joined_table) == len(df), 'row count changed'
assert abs(joined_table['revenue'].sum() - df['revenue'].sum()) < 0.00001, 'revenue total changed'
assert joined_table['vendor_name'].notna().all()

print(f"Rows Before/After: {len(df)} / {len(joined_table)}")
print(f"Revenue Before/After: ${df['revenue'].sum():,.2f} / ${joined_table['revenue'].sum():,.2f}")

joined_table.head()

# Interpretation: My left merge is safe (no rows added/lost or revenue changed)!

Unmatched Vendor Id: ['V-18']
Unmatched Order: 108
Unmatched Revenue: $2,349.00 (27.6% of total)
Rows Before/After: 400 / 400
Revenue Before/After: $8,520.00 / $8,520.00


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown (V-18)


**The unmatched vendor, and what I did about it:** The orders dataframe (called df in my code) has four vendor ids: V-01, V-05, V-10, and V-18. The vendor_names lookup table only has names for V-01, V-05, and V-10, so V-18 had no match when I merged. Those unmatched rows are 108 orders (worth $2,349.00), which is 27.6% of total revenue. Because V-18 is the biggest vendor by revenue, I didn't want to drop those rows. I used a left join to keep all the orders, then labeled the unmatched rows Unknown (V-18) so they still show up in the totals. Someone who owns the vendor list needs to supply V-18's real name, but the rows are still there so the totals aren't wrong when calculated.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pivot_table = pd.pivot_table(
    joined_table,
    index = 'vendor_name',
    columns = 'category',
    values = 'revenue',
    aggfunc = 'sum',
    fill_value = 0,
    margins = True,
    margins_name = 'Total Revenue',
)

display(pivot_table.round(2))

# Interpretation: The grand total in the bottom right cell is $8,520.0, which
# matches total revenue and shows me nothing was lost in the pivot. Because
# the data is displayed like this, we can easily see that Food is the biggest
# category in terms of total revenue, and Unknown (V-18) is the biggest vendor
# in terms of total revenue.

category,Drink,Food,Merch,RainGear,Total Revenue
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
Total Revenue,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined_table) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** One thing I would tell the vendors to do differently next game is to priortize selling food in order to make the most revenue. Food is their biggest seller by far: it brought in 4,293 dollars, which is 50.4% of the 8,520 dollar total. Seeing this, vendors should make sure that they never run out of food at games. RainGear was only 10.6% of revenue (901.50 dollars), so I would say it isn't worth stocking a lot of it (unless the game is predicted to be rainy). Drinks vary a lot between vendors: V-18 sold 582 dollars in drinks and Cav Merch North (V-10) sold 502.50 dollars, but Hoos Burgers (V-01) sold only 171 dollars even though it had the best Food sales (1,338 dollars). I would tell Hoos Burgers specifically that they should try promoting more drinks next game or taking a look at how they price drinks to see why they're falling behind in drink sales, but ahead of the game in food sales.

**b)** I would say that my answer for question 3 (highest average order revenue) is the least trustworthy. V-01 is first at 22.60 dollars per order over 94 orders, but V-10 is last at 20.31 dollars per order over 105 orders, a gap of only about 2 dollars. The vendors' average order revenue has such a small spread because every vendor sells the same kind of orders: each one has small orders as low as 4.50 dollars and big orders as high as 72.00 dollars, so their averages end up in the same range. Each vendor has only around 100 orders, so a few big orders could change who comes out on top. Because the gap between vendors (2 dollars) is small compared to how much individual orders vary (4.50 dollars to 72.00 dollars), I can't say that V-01 is definitively the best.